In [1]:
import sys
from pathlib import Path

# Add notebooks root to path for utils import
sys.path.insert(0, str(Path.cwd().parent))

from utils import compute_summary_stats, load_aggregate_results

SETUPS = ["copilot-opus-4-6", "gpt-5-2-codex"]
SETUP_LABELS = {"copilot-opus-4-6": "opus-4.6", "gpt-5-2-codex": "gpt-5.2"}

all_results = load_aggregate_results(category="bug-fix")

for setup in SETUPS:
    stats = compute_summary_stats(all_results[setup])
    print(f"{SETUP_LABELS[setup]:10s} | Runs: {stats['n_runs']} | Mean resolved: {stats['mean_resolved']:.1f}% | Avg time: {stats['avg_execution_time']:.0f}s")

opus-4.6   | Runs: 5 | Mean resolved: 65.1% | Avg time: 314s
gpt-5.2    | Runs: 5 | Mean resolved: 60.8% | Avg time: 195s


In [2]:
import pandas as pd

from bcbench.config import get_config
from bcbench.dataset import DatasetEntry, load_dataset_entries

_config = get_config()
bcbench_dataset: list[DatasetEntry] = load_dataset_entries(_config.paths.dataset_path)

dataset_df = pd.DataFrame(
    [
        {
            "instance_id": entry.instance_id,
            "image_count": entry.metadata.image_count or 0,
            "area": entry.metadata.area or "Unknown",
            "project": entry.extract_project_name() or "Unknown",
            "gold_patch": entry.patch,
        }
        for entry in bcbench_dataset
    ]
)

# Combine all setups into one dataframe with a "setup" column
merged_frames = []
for setup in SETUPS:
    df = all_results[setup].copy()
    df["setup"] = SETUP_LABELS[setup]
    merged_frames.append(df)

merged_df = pd.concat(merged_frames, ignore_index=True).merge(dataset_df, on="instance_id", how="left")
print(f"Merged {len(merged_df)} results ({len(merged_df) // len(SETUPS)} per setup) with dataset metadata")
merged_df.head(n=1)

Merged 1010 results (505 per setup) with dataset metadata


,run_id,instance_id,resolved,execution_time,setup,image_count,area,project,gold_patch
0,21815911073,microsoftInternal__NAV-204450,False,347.5,opus-4.6,9,finance,BaseApp,diff --git a/App/Layers/W1/BaseApp/Finance/VAT...


In [3]:
from unidiff import PatchSet


def count_files_in_patch(patch: str) -> int:
    return len(PatchSet(patch))


merged_df["expected_files"] = merged_df["gold_patch"].apply(count_files_in_patch)

bins = [0, 1, float("inf")]
labels = ["1", "2+"]
merged_df["files_bin"] = pd.cut(merged_df["expected_files"], bins=bins, labels=labels)

instance_df = merged_df.groupby(["setup", "instance_id"]).agg(files_bin=("files_bin", "first"), resolved_rate=("resolved", "mean")).reset_index()

pivot = instance_df.groupby(["files_bin", "setup"], observed=True).agg(pct=("resolved_rate", "mean"), count=("instance_id", "count")).reset_index()
pivot["pct"] = (pivot["pct"] * 100).round(1).astype(str) + "%"

result = pivot.pivot(index="files_bin", columns="setup", values="pct")
result["Tasks"] = pivot.groupby("files_bin", observed=True)["count"].first().values
result = result.rename_axis("Files Changed").reset_index()

total_instances = result["Tasks"].sum()
print('Average "% Resolved" by Number of Files Changed (gold patch):')
print(result.to_string(index=False))

Average "% Resolved" by Number of Files Changed (gold patch):
Files Changed gpt-5.2 opus-4.6  Tasks
            1   65.1%    70.2%     82
           2+   42.1%    43.2%     19


In [4]:
def count_loc_in_patch(patch: str) -> int:
    patch_set = PatchSet(patch)
    return patch_set.added + patch_set.removed


merged_df["expected_loc"] = merged_df["gold_patch"].apply(count_loc_in_patch)

bins = [0, 10, 25, float("inf")]
labels = ["1-10", "11-25", "26+"]
merged_df["loc_bin"] = pd.cut(merged_df["expected_loc"], bins=bins, labels=labels)

instance_df = merged_df.groupby(["setup", "instance_id"]).agg(loc_bin=("loc_bin", "first"), resolved_rate=("resolved", "mean")).reset_index()

pivot = instance_df.groupby(["loc_bin", "setup"], observed=True).agg(pct=("resolved_rate", "mean"), count=("instance_id", "count")).reset_index()
pivot["pct"] = (pivot["pct"] * 100).round(1).astype(str) + "%"

result = pivot.pivot(index="loc_bin", columns="setup", values="pct")
result["Tasks"] = pivot.groupby("loc_bin", observed=True)["count"].first().values
result = result.rename_axis("LoC Changed").reset_index()

print('Average "% Resolved" by Lines of Code Changed (gold patch):')
print(result.to_string(index=False))

Average "% Resolved" by Lines of Code Changed (gold patch):
LoC Changed gpt-5.2 opus-4.6  Tasks
       1-10   75.6%    78.5%     54
      11-25   38.3%    49.6%     23
        26+   49.2%    50.0%     24


In [5]:
merged_df["project_group"] = merged_df["project"].apply(lambda x: "BaseApp" if x == "BaseApp" else "First-party Apps")

instance_df = merged_df.groupby(["setup", "instance_id"]).agg(project_group=("project_group", "first"), resolved_rate=("resolved", "mean")).reset_index()

pivot = instance_df.groupby(["project_group", "setup"], observed=True).agg(pct=("resolved_rate", "mean"), count=("instance_id", "count")).reset_index()
pivot["pct"] = (pivot["pct"] * 100).round(1).astype(str) + "%"

result = pivot.pivot(index="project_group", columns="setup", values="pct")
result["Tasks"] = pivot.groupby("project_group", observed=True)["count"].first().values
result = result.rename_axis("Project Group").reset_index()

print('Average "% Resolved" by Project Group:')
print(result.to_string(index=False))

Average "% Resolved" by Project Group:
   Project Group gpt-5.2 opus-4.6  Tasks
         BaseApp   64.7%    67.1%     85
First-party Apps   40.0%    55.0%     16


In [6]:
bins = [-1, 0, 5, float("inf")]
labels = ["0", "1-5", "6+"]
merged_df["image_bin"] = pd.cut(merged_df["image_count"], bins=bins, labels=labels)

# Add problem statement char count
ps_chars = {entry.instance_id: len(entry.get_task(transform_image_paths=False)) for entry in bcbench_dataset}
merged_df["ps_chars"] = merged_df["instance_id"].map(ps_chars)

instance_df = (
    merged_df.groupby(["setup", "instance_id"])
    .agg(
        image_bin=("image_bin", "first"),
        resolved_rate=("resolved", "mean"),
        ps_chars=("ps_chars", "first"),
        expected_loc=("expected_loc", "first"),
    )
    .reset_index()
)

pivot = instance_df.groupby(["image_bin", "setup"], observed=True).agg(pct=("resolved_rate", "mean"), count=("instance_id", "count")).reset_index()
pivot["pct"] = (pivot["pct"] * 100).round(1).astype(str) + "%"

result = pivot.pivot(index="image_bin", columns="setup", values="pct")
result["Tasks"] = pivot.groupby("image_bin", observed=True)["count"].first().values

# Add avg chars and avg LoC per image bin (from first setup to avoid duplication)
first_setup = instance_df[instance_df["setup"] == SETUP_LABELS[SETUPS[0]]]
chars_by_bin = first_setup.groupby("image_bin", observed=True)["ps_chars"].mean().round(0).astype(int)
loc_by_bin = first_setup.groupby("image_bin", observed=True)["expected_loc"].mean().round(1)
result["Avg Chars"] = chars_by_bin.values
result["Avg LoC"] = loc_by_bin.values

result = result.rename_axis("Image Count").reset_index()

print('Average "% Resolved" by Image Count:')
print(result.to_string(index=False))

Average "% Resolved" by Image Count:
Image Count gpt-5.2 opus-4.6  Tasks  Avg Chars  Avg LoC
          0   53.5%    57.1%     34       1514     20.3
        1-5   56.7%    60.0%     30       1720     20.8
         6+   70.8%    76.8%     37       2118     16.1
